# CipherBase Colab RenderFree GPU rendering for CipherBase episodes1. Runtime > Change runtime type > GPU (T4)2. Run all cells3. MP4 downloads automatically

In [ ]:
# 1. Mount Drive + Clone repofrom google.colab import drivedrive.mount('/content/drive')import os, subprocessREPO_DIR = '/content/mythos'if not os.path.exists(REPO_DIR):    print('Cloning...')    subprocess.run(['git', 'clone', 'https://github.com/phanikumar2110/mythos.git', '/content/mythos'], check=True)else:    subprocess.run(['git', '-C', '/content/mythos', 'pull'], check=True)os.chdir('/content/mythos')print('Ready:', os.getcwd())

In [ ]:
# Install dependenciesimport subprocess, osprint('Installing system deps...')subprocess.run(['apt-get', 'update', '-qq'], check=True)subprocess.run(['apt-get', 'install', '-y', '-qq',    'ffmpeg', 'libcairo2-dev', 'libpango1.0-dev',    'libjpeg-dev', 'libgif-dev', 'librsvg2-dev',    'python3-dev', 'python3-venv', 'nodejs', 'npm',    'python3-cairo', 'python3-gi', 'gir1.2-gtk-3.0']    ], check=True)# Install Python packages with --break-system-packagesprint('Installing Python packages...')import subprocesssubprocess.run(['pip', 'install', '--break-system-packages', '-q',    'numpy', 'scipy', 'soundfile', 'cairocffi', 'pillow', 'tqdm'], check=True)subprocess.run(['pip', 'install', '--break-system-packages', '-q',    'kokoro-onnx', 'edge-tts', 'faster-whisper'], check=True)subprocess.run(['npm', 'install', '--prefix', '/content/mythos'] , check=True)print('Environment ready!')

In [ ]:
# RENDER EPISODE 1 - with full output capture
import json, subprocess, os, sys
from pathlib import Path
scene_path = Path('episodes/ep001_nexus_weaver.json')
with open(scene_path) as f:
    scene = json.load(f)
name = 'ep001'
out_dir = Path('episodes/ep001')
out_dir.mkdir(parents=True, exist_ok=True)
work_dir = Path('episodes/ep001/work')
work_dir.mkdir(exist_ok=True)
frame_dir = work_dir / 'frames'
frame_dir.mkdir(exist_ok=True)
print(f"Rendering {scene['meta']['title']} ({scene['meta']['duration']}s @ {scene['meta']['fps']}fps)")print(f"Total frames: {scene['meta']['duration'] * scene['meta']['fps']}")# 1. TTSprint('Generating narration...')import subprocess
result = subprocess.run([
    'python3', '-m', 'pip', 'run', '--break-system-packages', 'python', '-c',
    "import sys; sys.path.insert(0, '/content/mythos'); from tools.tts import generateTTS; generateTTS('episodes/ep001_nexus_weaver.json', 'episodes/ep001/work/narration')"
], check=True, cwd='/content/mythos', capture_output=True, text=True)
print('TTS stdout:', result.stdout[-500:])
if result.stderr:
    print('TTS stderr:', result.stderr[-500:])# 2. Render frames - CAPTURE ALL OUTPUTprint('Rendering frames...')duration = 300
fps = 24
total_frames = int(duration * 24)
print(f'Total frames to render: {total_frames}')import os
env = os.environ.copy()
env.update({    'EPISODE_JSON': 'episodes/ep001_nexus_weaver.json',    'FRAME_DIR': 'episodes/ep001/work/frames',    'W': '1920',    'H': '1080',    'FPS': '24',    'DURATION': '300'
})print('Starting render_full.js...')
result = subprocess.run([
    'node', 'render_full.js'
], cwd='/content/mythos', env=env, capture_output=True, text=True, timeout=3600)print('RETURN CODE:', result.returncode)print('STDOUT:', result.stdout[-2000:] if result.stdout else 'EMPTY')if result.stderr:
    print('STDERR:', result.stderr[-2000:])if result.returncode != 0:
    raise RuntimeError(f'Render failed with code {result.returncode}')# Check frames generatedframe_dir = Path('episodes/ep001/work/frames')
frames = list(frame_dir.glob('frame_*.png'))
print(f'Frames generated: {len(frames)}')if len(frames) == 0:
    raise RuntimeError('No frames generated!')

In [ ]:
# Encode videoimport subprocess
subprocess.run([
    'ffmpeg', '-y', '-framerate', '24',    '-i', 'episodes/ep001/work/frames/frame_%05d.png',    '-c:v', 'libx264', '-preset', 'medium', '-crf', '18',    '-pix_fmt', 'yuv420p',    'episodes/ep001/work/video_raw.mp4'
], check=True)subprocess.run([
    'ffmpeg', '-y',    '-i', 'episodes/ep001/work/video_raw.mp4',    '-c:v', 'copy',    '-an',    'episodes/ep001/ep001.mp4'
], check=True)print('Done! Downloading...')from google.colab import files
files.download('episodes/ep001/ep001.mp4')